In [26]:

import sys
!{sys.executable} -m pip install transformers datasets peft accelerate torch






[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model



In [28]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device == "cuda":
    print(torch.cuda.get_device_name(0))



Device: cpu


In [29]:

data = [
    {
        "text": "Input: Failed login attempts from same IP\nOutput: Threat: Brute Force Attack"
    },
    {
        "text": "Input: Suspicious email asking password\nOutput: Threat: Phishing Attack"
    },
    {
        "text": "Input: Large outbound traffic to unknown server\nOutput: Threat: Data Exfiltration"
    },
    {
        "text": "Input: Multiple port scans detected\nOutput: Threat: Reconnaissance Attack"
    },
    {
        "text": "Input: User downloaded ransomware attachment\nOutput: Threat: Malware Infection"
    }
]

dataset = Dataset.from_list(data)
print(dataset)



Dataset({
    features: ['text'],
    num_rows: 5
})


In [30]:

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)

print("Model Loaded")


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7599.64it/s]


Model Loaded


In [31]:

def tokenize(example):
    encoded = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    encoded["labels"] = encoded["input_ids"].copy()
    return encoded

tokenized_dataset = dataset.map(tokenize)
print(tokenized_dataset[0])



Map: 100%|██████████| 5/5 [00:00<00:00, 1648.45 examples/s]

{'text': 'Input: Failed login attempts from same IP\nOutput: Threat: Brute Force Attack', 'input_ids': [20560, 25, 22738, 17594, 6370, 422, 976, 6101, 198, 26410, 25, 25238, 25, 1709, 1133, 5221, 8307, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [20560, 25, 22738, 17594, 6370, 422, 976, 6101, 198, 26410, 25, 25238, 25, 1709, 1133, 5221, 8307, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 

In [32]:

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



trainable params: 73,728 || all params: 81,986,304 || trainable%: 0.0899


In [33]:

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=20,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=False
)


In [34]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)



In [35]:

trainer.train()



prompt = "Input: Suspicious email asking password\nOutput:"

inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

output = model.generate(
    **inputs,
    max_new_tokens=20
)

print(tokenizer.decode(output[0], skip_special_tokens=True))



Step,Training Loss
1,8.934902
2,8.522277
3,8.686410
4,8.869995
5,8.870255
6,8.715184
7,8.636578
8,8.431909
9,8.833028
10,8.342169


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Input: Suspicious email asking password
Output:






















In [36]:

model.save_pretrained("./cyber_lora")
tokenizer.save_pretrained("./cyber_lora")

print("Adapter Saved")



from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("distilgpt2")

loaded_model = PeftModel.from_pretrained(
    base_model,
    "./cyber_lora"
)

loaded_model.to(device)


Adapter Saved


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 8231.56it/s]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-5): 6 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
  

In [37]:



prompt = "Input: Failed login attempts from same IP\nOutput:"

inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

output = loaded_model.generate(
    **inputs,
    max_new_tokens=20
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


import os
print(os.listdir("./cyber_lora"))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Input: Failed login attempts from same IP
Output:




















['adapter_config.json', 'adapter_model.safetensors', 'README.md', 'tokenizer.json', 'tokenizer_config.json']
